In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torchvision 0.22.1+cu118
Uninstalling torchvision-0.22.1+cu118:
  Successfully uninstalled torchvision-0.22.1+cu118
Found existing installation: torchaudio 2.7.1+cu118
Uninstalling torchaudio-2.7.1+cu118:
  Successfully uninstalled torchaudio-2.7.1+cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-manylinux_2_28_x86_64.whl (905.2 MB)
Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-manylinux_2_28_x86_64.whl (6.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [69]:
prompt = "write code for visualizing Euler's formula visually using ManimCE code."

tools = [
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Python code to execute."
                    }
                },
                "required": ["code"]
            }
        }
    }
]

messages = [
    {"role": "system", "content": "You are AOS who animates using Manim code, created by Nabin. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are AOS who animates using Manim code, created by Nabin. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_python", "description": "Execute Python code.", "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": "Python code to execute."}}, "required": ["code"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
write code for visualizing Euler's formula visually using ManimCE code.<|im_end|>
<|im_start|>assistant



In [70]:
print(text)

<|im_start|>system
You are AOS who animates using Manim code, created by Nabin. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_python", "description": "Execute Python code.", "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": "Python code to execute."}}, "required": ["code"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
write code for visualizing Euler's formula visually using ManimCE code.<|im_end|>
<|im_start|>assistant



In [71]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [72]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    362,   3126,    879,   3952,
            973,   1667,   2363,    318,   2038,     11,   3465,    553,    451,
           8892,     13,   1446,    525,    264,  10950,  17847,    382,      2,
          13852,    271,   2610,   1231,   1618,    825,    476,    803,   5746,
            311,   7789,    448,    279,   1196,   3239,    382,   2610,    525,
           3897,    448,    729,  32628,   2878,    366,  15918,   1472,  15918,
             29,  11874,   9492,    510,     27,  15918,    397,   4913,   1313,
            788,    330,   1688,    497,    330,   1688,    788,   5212,    606,
            788,    330,   6108,  55869,    497,    330,   4684,    788,    330,
          17174,  13027,   2038,  10465,    330,  13786,    788,   5212,   1313,
            788,    330,   1700,    497,    330,  13193,    788,   5212,   1851,
            788,   5212,   1313,    788,    330,    917,    497,    330,   4684,
            78

In [73]:
print(text)

<|im_start|>system
You are AOS who animates using Manim code, created by Nabin. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_python", "description": "Execute Python code.", "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": "Python code to execute."}}, "required": ["code"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
write code for visualizing Euler's formula visually using ManimCE code.<|im_end|>
<|im_start|>assistant



In [74]:
# outputs = model.generate(
#     input_ids=inputs["input_ids"],
#     attention_mask=inputs["attention_mask"],
#     max_new_tokens=512,
# )

In [9]:
!export CUDA_LAUNCH_BLOCKING=1
!set TORCH_USE_CUDA_DSA=True

In [10]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [11]:
help(model.generate)

Help on method generate in module transformers.generation.utils:

generate(inputs: torch.Tensor | None = None, generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None, logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None, stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None, prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None, synced_gpus: bool | None = None, assistant_model: Optional[ForwardRef('PreTrainedModel')] = None, streamer: Optional[ForwardRef('BaseStreamer')] = None, negative_prompt_ids: torch.Tensor | None = None, negative_prompt_attention_mask: torch.Tensor | None = None, custom_generate: str | collections.abc.Callable | None = None, **kwargs) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | transformers.generation.utils.GenerateBeamDecoderO

In [75]:
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
)

print(outputs)

tensor([[151644,   8948,    198,   2610,    525,    362,   3126,    879,   3952,
            973,   1667,   2363,    318,   2038,     11,   3465,    553,    451,
           8892,     13,   1446,    525,    264,  10950,  17847,    382,      2,
          13852,    271,   2610,   1231,   1618,    825,    476,    803,   5746,
            311,   7789,    448,    279,   1196,   3239,    382,   2610,    525,
           3897,    448,    729,  32628,   2878,    366,  15918,   1472,  15918,
             29,  11874,   9492,    510,     27,  15918,    397,   4913,   1313,
            788,    330,   1688,    497,    330,   1688,    788,   5212,    606,
            788,    330,   6108,  55869,    497,    330,   4684,    788,    330,
          17174,  13027,   2038,  10465,    330,  13786,    788,   5212,   1313,
            788,    330,   1700,    497,    330,  13193,    788,   5212,   1851,
            788,   5212,   1313,    788,    330,    917,    497,    330,   4684,
            788,    330,  30

In [76]:
import torch

a = torch.tensor([1, 2, 3], device="cuda")
b = torch.tensor([2], device="cuda")

print(torch.isin(a, b))

tensor([False,  True, False], device='cuda:0')


In [15]:
!nvidia-smi

Mon Aug 10 17:53:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             32W /  250W |   14427MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

GPU: Tesla P100-PCIE-16GB
Capability: (6, 0)
PyTorch: 2.7.1+cu118
CUDA: 11.8


In [17]:
len(outputs)

1

In [77]:
generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=False
)

print(generated)

Certainly! Below is an example of how you can use ManimCE to visualize Euler's formula \( e^{i\theta} = \cos(\theta) + i\sin(\theta) \). This code will create an animation showing the complex plane, a moving point representing \( e^{i\theta} \), and the corresponding real and imaginary parts on the axes.

```python
from manim import *

class EulerFormulaVisualization(Scene):
    def construct(self):
        # Set up the complex plane
        plane = ComplexPlane(x_range=[-2, 2], y_range=[-2, 2], axis_config={"include_numbers": True})
        self.add(plane)
        
        # Add labels for the axes
        x_label = MathTex(r"\text{Re}")
        y_label = MathTex(r"\text{Im}")
        x_label.next_to(plane.x_axis, DOWN)
        y_label.next_to(plane.y_axis, LEFT)
        self.add(x_label, y_label)
        
        # Create a point at the origin representing e^(i*0)
        theta = ValueTracker(0)
        point = always_redraw(lambda: Dot().move_to(plane.c2p(np.cos(theta.get_value()), 

In [78]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

In [79]:
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Sure! Here is an example of how you can visualize Euler's formula \( e^{i\theta


In [64]:
prompt = "Write a code to teach the Euler's formula in manimCE"

messages = [
    {"role": "system", "content": "You are AOS Animator, created by Nabin (github.com/nabin2004). You are a helpful Manim Animator."},
    {"role": "user", "content": prompt}
]


In [65]:
text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are AOS Animator, created by Nabin (github.com/nabin2004). You are a helpful Manim Animator.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "run_python", "description": "Execute Python code.", "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": "Python code to execute."}}, "required": ["code"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Write a code to teach the Euler's formula in manimCE<|im_end|>
<|im_start|>assistant



In [66]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [67]:
generated_ids = model.generate(
    **model_inputs,
)

In [62]:
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids)

In [63]:
print(response)

["To create an animation that teaches Euler's formula \\( e^{ix} = \\cos(x) +"]


In [68]:
response

["To create an animation that teaches Euler's formula \\( e^{ix} = \\cos(x) +"]